# Hierarchical Models with minibayes

This notebook demonstrates hierarchical (multilevel) models using minibayes.

**Key concepts:**
- **Vector parameters**: Parameters with multiple components (e.g., group-specific effects)
- **Conditional priors**: Distributions whose hyperparameters depend on other parameters
- **Partial pooling**: Sharing information across groups while allowing group-specific variation

## The Eight Schools Problem

A classic example from Rubin (1981): Eight schools conducted SAT coaching experiments. Each school reported an estimated treatment effect and standard error. The question: what can we learn about the effectiveness of coaching programs?

**Data:**
- $y_j$: Estimated treatment effect for school $j$
- $\sigma_j$: Standard error of the estimate (known)

**Hierarchical Model:**
$$\mu \sim N(0, 5)$$
$$\tau \sim \text{HalfNormal}(5)$$
$$\theta_j \sim N(\mu, \tau) \quad \text{for } j = 1, \ldots, 8$$
$$y_j \sim N(\theta_j, \sigma_j)$$

Where:
- $\mu$: Overall mean effect
- $\tau$: Between-school standard deviation
- $\theta_j$: True effect for school $j$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import minibayes as mb
from minibayes import Model, dist, viz
from minibayes.diagnostics import summary

In [ ]:
# Eight Schools data (Rubin, 1981)
school_names = ["A", "B", "C", "D", "E", "F", "G", "H"]
y = np.array([28.0, 8.0, -3.0, 7.0, -1.0, 1.0, 18.0, 12.0])  # Estimated effects
sigma = np.array([15.0, 10.0, 16.0, 11.0, 9.0, 11.0, 10.0, 18.0])  # Standard errors

J = len(y)  # Number of schools

print("Eight Schools Data:")
print("-" * 40)
for j, name in enumerate(school_names):
    print(f"School {name}: effect = {y[j]:5.1f} +/- {sigma[j]:.1f}")

In [ ]:
# Visualize the raw data
with viz.style():
    fig, ax = plt.subplots(figsize=(10, 5))

    ax.errorbar(school_names, y, yerr=1.96 * sigma, fmt='o', capsize=5,
                color=viz.PALETTE["terracotta"], markersize=8, label="Observed (95% CI)")
    ax.axhline(0, color="gray", linestyle="--", alpha=0.5)
    ax.axhline(np.mean(y), color=viz.PALETTE["sage"], linestyle=":", lw=2,
               label=f"Pooled mean = {np.mean(y):.1f}")

    ax.set_xlabel("School")
    ax.set_ylabel("Treatment Effect")
    ax.set_title("Eight Schools: Observed Effects with 95% CI")
    ax.legend()
    plt.show()

print("\nNote: Wide confidence intervals suggest high uncertainty in individual estimates.")
print("A hierarchical model can improve estimates by partial pooling.")

---
## Define the Hierarchical Model

Using the `p(...)` API, we define priors where:
- `mu` and `tau` are hyperparameters
- `theta` is a vector parameter with `size=8`, where each element is drawn from `N(mu, tau)`

The key insight: when `p("theta", dist.Normal(mu, tau), size=8)` is called, `mu` and `tau` are already sampled floats, so `Normal(mu, tau)` creates a distribution with those specific values.

In [ ]:
def priors(p):
    """Hierarchical prior for Eight Schools."""
    # Hyperparameters
    mu = p("mu", dist.Normal(0, 5))       # Overall mean
    tau = p("tau", dist.HalfNormal(5))    # Between-school std

    # School-specific effects (vector parameter)
    # Each theta[j] ~ N(mu, tau) - conditional on mu and tau
    theta = p("theta", dist.Normal(mu, tau), size=J)


def log_likelihood(params, data):
    """Normal likelihood for observed effects."""
    y_obs, sigma_obs = data
    theta = params["theta"]  # shape (8,)

    # y[j] ~ N(theta[j], sigma[j])
    # Sum log_prob over all schools
    total = 0.0
    for j in range(len(y_obs)):
        total += dist.Normal(theta[j], sigma_obs[j]).log_prob(y_obs[j])
    return total


model = Model(priors=priors, log_likelihood=log_likelihood)

print(f"Parameters: {model.param_names}")
print(f"Flat parameters: {model.flat_param_names}")
print(f"\nParameter info:")
for name, info in model.param_info.items():
    print(f"  {name}: is_vector={info.is_vector}, size={info.size}")

---
## Prior Predictive Check

Before fitting, let's check that our priors generate reasonable data ranges.

In [ ]:
# Sample from prior
def predictive_fn(params, rng):
    theta = params["theta"]
    # Simulate observed data
    y_sim = np.array([dist.Normal(theta[j], sigma[j]).sample(rng=rng) for j in range(J)])
    return {"y_sim": y_sim, "theta": theta}

prior_ppc = mb.sample_prior_predictive(model, predictive_fn, num_samples=200, seed=42)

# Plot prior predictive
with viz.style():
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    # Left: Distribution of theta values
    ax = axes[0]
    for j in range(J):
        ax.hist(prior_ppc["theta"][:, j], bins=30, alpha=0.3, label=f"School {school_names[j]}")
    ax.set_xlabel("theta (true effect)")
    ax.set_ylabel("Count")
    ax.set_title("Prior: School Effects")
    ax.set_xlim(-30, 30)

    # Right: Simulated observations vs actual
    ax = axes[1]
    for i in range(0, 200, 5):
        ax.scatter(range(J), prior_ppc["y_sim"][i], alpha=0.08, color=viz.PALETTE["blue"], s=20)
    ax.errorbar(range(J), y, yerr=1.96 * sigma, fmt='o', capsize=5,
                color=viz.PALETTE["terracotta"], markersize=8, label="Observed", zorder=10)
    ax.set_xticks(range(J))
    ax.set_xticklabels(school_names)
    ax.set_xlabel("School")
    ax.set_ylabel("Effect")
    ax.set_title("Prior Predictive vs Observed")
    ax.legend()

    plt.tight_layout()
    plt.show()

print("Prior predictive covers the observed data range - priors are reasonable.")

---
## Run MCMC Sampling

In [ ]:
# Run MCMC with Adaptive MH
result = mb.sample(
    model,
    data=(y, sigma),
    num_samples=50000,
    num_warmup=5000,
    num_chains=2,
    sampler="adaptive_mh",
    seed=42,
    progress=True,
)

print(f"\\nSampling complete in {result.elapsed_time:.2f}s")
for i, rate in enumerate(result.acceptance_rate):
    print(f"Chain {i+1}: acceptance rate = {rate:.1%}")

In [ ]:
# Summary statistics
print(viz.summary_table(result))

---
## Visualize Results

In [ ]:
# Trace plots for hyperparameters
fig = viz.plot_samples(result, params=["mu", "tau"])
plt.suptitle("Hyperparameter Traces", y=1.02)
plt.show()

In [ ]:
# Density plots for hyperparameters
fig = viz.plot_density(result, params=["mu", "tau"])
plt.show()

In [ ]:
# Shrinkage plot: compare raw estimates to posterior estimates
theta_samples = result.samples["theta"]  # shape: (chains, samples, 8)
theta_mean = theta_samples.mean(axis=(0, 1))  # Mean over chains and samples
theta_std = theta_samples.std(axis=(0, 1))

mu_mean = result.samples["mu"].mean()

with viz.style():
    fig, ax = plt.subplots(figsize=(10, 6))

    # Raw estimates with error bars
    ax.errorbar(range(J), y, yerr=1.96 * sigma, fmt='s', capsize=5,
                color=viz.PALETTE["terracotta"], markersize=10,
                label="Raw estimate (95% CI)", alpha=0.7)

    # Posterior estimates with credible intervals
    ax.errorbar(np.arange(J) + 0.15, theta_mean, yerr=1.96 * theta_std, fmt='o', capsize=5,
                color=viz.PALETTE["blue"], markersize=10,
                label="Posterior (95% CI)")

    # Overall mean
    ax.axhline(mu_mean, color=viz.PALETTE["sage"], linestyle="--", lw=2,
               label=f"Overall mean (mu) = {mu_mean:.1f}")

    # Shrinkage arrows
    for j in range(J):
        ax.annotate("", xy=(j + 0.15, theta_mean[j]), xytext=(j, y[j]),
                    arrowprops=dict(arrowstyle="->", color="gray", alpha=0.5))

    ax.set_xticks(range(J))
    ax.set_xticklabels(school_names)
    ax.set_xlabel("School")
    ax.set_ylabel("Treatment Effect")
    ax.set_title("Shrinkage: Raw Estimates vs Posterior Estimates")
    ax.legend(loc="upper right")
    plt.show()

print("\nShrinkage effect: Extreme estimates are pulled toward the overall mean.")
print("This is 'partial pooling' - borrowing strength across schools.")

In [ ]:
# Forest plot for all school effects
with viz.style():
    fig, ax = plt.subplots(figsize=(8, 6))

    # Get quantiles for each school
    y_pos = np.arange(J)

    for j in range(J):
        samples_j = theta_samples[:, :, j].flatten()
        q5, q50, q95 = np.percentile(samples_j, [5, 50, 95])

        # Credible interval
        ax.plot([q5, q95], [j, j], color=viz.PALETTE["blue"], lw=2)
        # Median
        ax.scatter([q50], [j], color=viz.PALETTE["blue"], s=80, zorder=5)
        # Raw estimate
        ax.scatter([y[j]], [j], color=viz.PALETTE["terracotta"], s=60, marker='s', alpha=0.7)

    # Overall mean
    ax.axvline(mu_mean, color=viz.PALETTE["sage"], linestyle="--", lw=2, label=f"mu = {mu_mean:.1f}")
    ax.axvline(0, color="gray", linestyle=":", alpha=0.5)

    ax.set_yticks(y_pos)
    ax.set_yticklabels([f"School {name}" for name in school_names])
    ax.set_xlabel("Treatment Effect")
    ax.set_title("Posterior School Effects (90% CI)")
    ax.legend(loc="lower right")

    # Add legend for markers
    ax.scatter([], [], color=viz.PALETTE["blue"], s=80, label="Posterior median")
    ax.scatter([], [], color=viz.PALETTE["terracotta"], s=60, marker='s', label="Raw estimate")
    ax.legend(loc="lower right")

    plt.tight_layout()
    plt.show()

---
## Posterior Predictive Check

In [ ]:
# Posterior predictive
def post_predictive_fn(params, rng):
    theta = params["theta"]
    y_rep = np.array([dist.Normal(theta[j], sigma[j]).sample(rng=rng) for j in range(J)])
    return {"y_rep": y_rep}

ppc = mb.sample_posterior_predictive(result, post_predictive_fn, num_samples=500, seed=42)

# Plot posterior predictive
with viz.style():
    fig, ax = plt.subplots(figsize=(10, 5))

    # Posterior predictive samples (plot subset)
    for i in range(0, 500, 5):
        ax.scatter(range(J), ppc["y_rep"][i], alpha=0.03, color=viz.PALETTE["blue"], s=30)

    # Observed data
    ax.scatter(range(J), y, color=viz.PALETTE["terracotta"], s=100, zorder=10,
               edgecolor="white", linewidth=2, label="Observed")

    ax.set_xticks(range(J))
    ax.set_xticklabels(school_names)
    ax.set_xlabel("School")
    ax.set_ylabel("Treatment Effect")
    ax.set_title("Posterior Predictive Check")
    ax.legend()
    plt.show()

print("Observed data falls within the posterior predictive distribution - model fits well.")

---
## Key Takeaways

### Hierarchical Models in minibayes

1. **Vector parameters**: Use `size=` to create parameters with multiple components
   ```python
   theta = p("theta", dist.Normal(mu, tau), size=8)
   ```

2. **Conditional priors**: Distributions can depend on other parameters
   - Execution order defines dependencies
   - No special syntax needed - just use the parameter values

3. **Partial pooling**: Hierarchical models shrink extreme estimates toward the group mean
   - More shrinkage for uncertain estimates (large $\sigma_j$)
   - Less shrinkage for precise estimates (small $\sigma_j$)

### Results Shape

- Scalar parameters: `(num_chains, num_samples)`
- Vector parameters: `(num_chains, num_samples, size)`

```python
result.samples["mu"].shape    # (4, 5000)
result.samples["theta"].shape # (4, 5000, 8)
```